#### TextCNN
- CNN(합성곱 신경망) : 이미지 분석시 사용하는 신경망 모델 
- 이미지에서 사용이 되는 CNN을 이용하여 텍스트를 이미지와 같다 라고 가정을 하고 설계한 모델 
1. 문장의 행렬화 (임베딩)
    - 배치로 문장을 모은다. ( 배치사이즈, 문장의 길이(토큰 개수) )
    - 임베딩 처리를 통해서 ( 배치사이즈, 문장의 길이, 임베딩 차원의 수 )
2. 텍스트 돋보기로 문맥을 읽기 
    - 단어의 흐름 방향으로 구간은 선택하여 데이터 학습 
3. 맥스풀링(가장 큰 값을 찾는 과정) 
    - 2번 과정에서의 구간 데이터들을 합성곱을 통해 가장 큰 값을 선택하는 과정 
    - 가장 강한 인상 남기기
4. 최종 분류 
    - 해당 데이터셋을 이용하여 최종 분류 하는 과정 
    - kim CNN 구조는 단어를 3, 4, 5로 분류하여 맥스풀링을 한 뒤 분류 
    - 과적합의 위험성 때문에 dropout()을 이용하여 일정 피쳐를 0으로 만들어서 과적합 방지 

#### TextCNN 사용하기 전 데이터를 준비 
1. 데이터 로드 
2. 데이터 튜닝 
3. 데이터 토큰화 
4. 단어 사전 등록
5. 단어 사전을 이용한 인코딩 

In [114]:
import pandas as pd 
import re 
from konlpy.tag import Komoran
from collections import Counter

In [115]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [116]:
df.dropna(inplace=True)

In [117]:
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', str(text))
    text = re.sub(r"\s+", ' ', text).strip()
    return text

In [118]:
df['document'] = df['document'].map(normalize)

In [119]:
df.info()

<class 'pandas.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        149995 non-null  int64
 1   document  149995 non-null  str  
 2   label     149995 non-null  int64
dtypes: int64(2), str(1)
memory usage: 4.6 MB


In [120]:
# 빈 텍스트 제외 
df = df.loc[
    ~(df['document'] == ''), 
]

In [121]:
# 중복 데이터 제거 
df.drop_duplicates('document', inplace=True)

In [122]:
komoran = Komoran()

allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XR']
stop_word = ['하다', '되다', '이다']

def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if word not in stop_word and pos in allow_pos:
            tokens.append(word)
    return tokens

In [123]:
df2 = df[:1000]

In [124]:
texts, labels = df2['document'].values, df2['label'].values

# texts 토큰화 
tokens_list = [tokenize(text) for text in texts]
tokens_list

[['더빙', '진짜', '짜증', '나', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없', '평점', '조정'],
 ['익살', '연기', '돋보이', '영화', '스파이더맨', '늙', '보이', '하', '커스틴 던스트', '너무나'],
 ['막', '걸음마', '떼', '초등학교', '학년', '용', '영화', '별', '반개', '아깝'],
 ['원작', '긴장감', '제대로', '살리'],
 ['반개',
  '아깝',
  '욕',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '이',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '이',
  '드라마',
  '가족',
  '없',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '없', '재미', '있', '안', '영화'],
 ['왜', '평점', '낮', '꽤', '보', '헐리우드', '화려', '너무', '길들이', '있'],
 [],
 ['볼', '때', '눈물', '나서', '죽', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['울', '손들', '횡단보도', '건너', '때', '뛰쳐나오', '이범수', '연기', '드럽'],
 ['담백', '깔끔', '좋', '신문', '기사', '로만', '보다', '보', '자꾸', '잊어버리', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보',
  '영화',
  '가장',
  '노',
  '재',
  '노',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['참',
  '사람',
  '웃기',
  '바스코',
  '이기',
  '락스',
  '코',
  '까',
  '고',
  '바비',
  '이기',
  '아이

In [125]:
# 최소 등장 횟수에 따른 단어 사전을 생성 
Counter(word for toks in tokens_list for word in toks)

Counter({'영화': 364,
         '보': 262,
         '하': 133,
         '없': 114,
         '있': 86,
         '좋': 73,
         '진짜': 68,
         '정말': 67,
         '안': 66,
         '너무': 61,
         '연기': 58,
         '재밌': 57,
         '같': 56,
         '나오': 52,
         '만들': 51,
         '잘': 47,
         '최고': 47,
         '다': 46,
         '사람': 44,
         '왜': 43,
         '되': 41,
         '알': 38,
         '내용': 36,
         '때': 34,
         '감동': 34,
         '말': 34,
         '더': 33,
         '재미없': 32,
         '재미': 31,
         '아깝': 31,
         '배우': 31,
         '재미있': 30,
         '생각': 30,
         '시간': 29,
         '드라마': 28,
         '그냥': 28,
         '좀': 28,
         '평점': 27,
         '스토리': 27,
         '감독': 27,
         '나': 26,
         '이': 26,
         '쓰레기': 26,
         '작품': 25,
         '모르': 24,
         '완전': 23,
         '지루': 22,
         '들': 22,
         '느낌': 21,
         '보이': 20,
         '남': 20,
         '정도': 20,
         '다시': 19,
    

In [126]:
t_s = []
for toks in tokens_list:
    for word in toks:
        t_s.append(word)
freq = Counter(t_s)

In [127]:
freq.items()

dict_items([('더빙', 5), ('진짜', 68), ('짜증', 16), ('나', 26), ('목소리', 2), ('포스터', 8), ('초딩', 3), ('영화', 364), ('오버', 3), ('연기', 58), ('가볍', 3), ('교도소', 1), ('이야기', 14), ('솔직히', 14), ('재미', 31), ('없', 114), ('평점', 27), ('조정', 1), ('익살', 1), ('돋보이', 3), ('스파이더맨', 1), ('늙', 3), ('보이', 20), ('하', 133), ('커스틴 던스트', 1), ('너무나', 7), ('막', 8), ('걸음마', 1), ('떼', 3), ('초등학교', 4), ('학년', 3), ('용', 2), ('별', 2), ('반개', 3), ('아깝', 31), ('원작', 6), ('긴장감', 6), ('제대로', 2), ('살리', 5), ('욕', 7), ('나오', 52), ('이응경', 1), ('길용우', 1), ('생활', 2), ('이', 26), ('정말', 67), ('발로', 2), ('납치', 1), ('감금', 1), ('반복', 3), ('드라마', 28), ('가족', 6), ('못하', 3), ('사람', 44), ('모이', 1), ('액션', 18), ('있', 86), ('안', 66), ('왜', 43), ('낮', 8), ('꽤', 8), ('보', 262), ('헐리우드', 2), ('화려', 5), ('너무', 61), ('길들이', 1), ('볼', 12), ('때', 34), ('눈물', 6), ('나서', 6), ('죽', 14), ('향수', 2), ('자극', 5), ('허진호', 1), ('감성', 3), ('절제', 3), ('멜로', 6), ('달인', 1), ('울', 9), ('손들', 1), ('횡단보도', 1), ('건너', 1), ('뛰쳐나오', 1), ('이범수', 1), ('드럽', 4), ('담백', 2),

In [128]:
# 최소 등장 횟수는 2회 
min_count = 2

# 단어 사전에 특수 토큰 <PAD> <UNK> 토큰을 먼저 대입 
vocab = ['<PAD>', '<UNK>']

for word, cnt in freq.items():
    # word : 단어
    # cnt : 출현 횟수
    if cnt >= min_count:
        vocab.append(word)

In [129]:
len(vocab)

1046

In [130]:
stoi = {word : idx for idx, word in enumerate(vocab)}
stoi

{'<PAD>': 0,
 '<UNK>': 1,
 '더빙': 2,
 '진짜': 3,
 '짜증': 4,
 '나': 5,
 '목소리': 6,
 '포스터': 7,
 '초딩': 8,
 '영화': 9,
 '오버': 10,
 '연기': 11,
 '가볍': 12,
 '이야기': 13,
 '솔직히': 14,
 '재미': 15,
 '없': 16,
 '평점': 17,
 '돋보이': 18,
 '늙': 19,
 '보이': 20,
 '하': 21,
 '너무나': 22,
 '막': 23,
 '떼': 24,
 '초등학교': 25,
 '학년': 26,
 '용': 27,
 '별': 28,
 '반개': 29,
 '아깝': 30,
 '원작': 31,
 '긴장감': 32,
 '제대로': 33,
 '살리': 34,
 '욕': 35,
 '나오': 36,
 '생활': 37,
 '이': 38,
 '정말': 39,
 '발로': 40,
 '반복': 41,
 '드라마': 42,
 '가족': 43,
 '못하': 44,
 '사람': 45,
 '액션': 46,
 '있': 47,
 '안': 48,
 '왜': 49,
 '낮': 50,
 '꽤': 51,
 '보': 52,
 '헐리우드': 53,
 '화려': 54,
 '너무': 55,
 '볼': 56,
 '때': 57,
 '눈물': 58,
 '나서': 59,
 '죽': 60,
 '향수': 61,
 '자극': 62,
 '감성': 63,
 '절제': 64,
 '멜로': 65,
 '울': 66,
 '드럽': 67,
 '담백': 68,
 '깔끔': 69,
 '좋': 70,
 '기사': 71,
 '보다': 72,
 '자꾸': 73,
 '취향': 74,
 '극장': 75,
 '가장': 76,
 '노': 77,
 '재': 78,
 '감동': 79,
 '스토리': 80,
 '어거지': 81,
 '참': 82,
 '웃기': 83,
 '이기': 84,
 '코': 85,
 '까': 86,
 '고': 87,
 '깔': 88,
 '그냥': 89,
 '난': 90,
 '이해': 91,
 '뒤': 

In [131]:
stoi2 = dict()
for idx, word in enumerate(vocab):
    # idx : 위치 값
    # word : 단어 
    stoi2[word] = idx

stoi2

{'<PAD>': 0,
 '<UNK>': 1,
 '더빙': 2,
 '진짜': 3,
 '짜증': 4,
 '나': 5,
 '목소리': 6,
 '포스터': 7,
 '초딩': 8,
 '영화': 9,
 '오버': 10,
 '연기': 11,
 '가볍': 12,
 '이야기': 13,
 '솔직히': 14,
 '재미': 15,
 '없': 16,
 '평점': 17,
 '돋보이': 18,
 '늙': 19,
 '보이': 20,
 '하': 21,
 '너무나': 22,
 '막': 23,
 '떼': 24,
 '초등학교': 25,
 '학년': 26,
 '용': 27,
 '별': 28,
 '반개': 29,
 '아깝': 30,
 '원작': 31,
 '긴장감': 32,
 '제대로': 33,
 '살리': 34,
 '욕': 35,
 '나오': 36,
 '생활': 37,
 '이': 38,
 '정말': 39,
 '발로': 40,
 '반복': 41,
 '드라마': 42,
 '가족': 43,
 '못하': 44,
 '사람': 45,
 '액션': 46,
 '있': 47,
 '안': 48,
 '왜': 49,
 '낮': 50,
 '꽤': 51,
 '보': 52,
 '헐리우드': 53,
 '화려': 54,
 '너무': 55,
 '볼': 56,
 '때': 57,
 '눈물': 58,
 '나서': 59,
 '죽': 60,
 '향수': 61,
 '자극': 62,
 '감성': 63,
 '절제': 64,
 '멜로': 65,
 '울': 66,
 '드럽': 67,
 '담백': 68,
 '깔끔': 69,
 '좋': 70,
 '기사': 71,
 '보다': 72,
 '자꾸': 73,
 '취향': 74,
 '극장': 75,
 '가장': 76,
 '노': 77,
 '재': 78,
 '감동': 79,
 '스토리': 80,
 '어거지': 81,
 '참': 82,
 '웃기': 83,
 '이기': 84,
 '코': 85,
 '까': 86,
 '고': 87,
 '깔': 88,
 '그냥': 89,
 '난': 90,
 '이해': 91,
 '뒤': 

In [132]:
# 인코딩 함수 작성 -> torch의 기본 데이터 형태 tensor형으로 변환 
import torch

In [133]:
def encode(toks):
    result = [stoi.get(word, stoi['<UNK>']) for word in toks]

    # result = []
    # for word in toks:
    #     result.append(
    #         stoi.get(word, stoi['<UNK>'])
    #     )

    return torch.tensor(result, dtype=torch.long)

In [134]:
enc_inputs = [ encode(toks) for toks in tokens_list ]
enc_inputs

[tensor([2, 3, 4, 5, 6]),
 tensor([ 7,  8,  9, 10, 11, 12]),
 tensor([], dtype=torch.int64),
 tensor([ 1, 13, 14, 15, 16, 17,  1]),
 tensor([ 1, 11, 18,  9,  1, 19, 20, 21,  1, 22]),
 tensor([23,  1, 24, 25, 26, 27,  9, 28, 29, 30]),
 tensor([31, 32, 33, 34]),
 tensor([29, 30, 35, 36,  1,  1, 11, 37, 38, 39, 40,  1,  1, 41, 41, 38, 42, 43,
         16, 11, 44, 45,  1]),
 tensor([46, 16, 15, 47, 48,  9]),
 tensor([49, 17, 50, 51, 52, 53, 54, 55,  1, 47]),
 tensor([], dtype=torch.int64),
 tensor([56, 57, 58, 59, 60, 61, 62,  1, 63, 64, 65,  1]),
 tensor([66,  1,  1,  1, 57,  1,  1, 11, 67]),
 tensor([68, 69, 70,  1, 71,  1, 72, 52, 73,  1, 45]),
 tensor([74,  1,  3, 75, 52,  9, 76, 77, 78, 77, 79, 80, 81, 79, 81]),
 tensor([1, 1]),
 tensor([82, 45, 83,  1, 84,  1, 85, 86, 87,  1, 84,  1, 88, 89, 86,  1, 90, 20]),
 tensor([ 1,  1, 91, 49, 92, 93, 94]),
 tensor([ 95,  39,  96,  97,   1,   1,  98,  99, 100,   1, 101,   1,  96, 102]),
 tensor([  1, 103,   1, 104, 105]),
 tensor([  1, 106,  4

In [135]:
# label 데이터도 tensor 형태로 변환 
label_t = torch.tensor(labels, dtype=torch.long)

In [136]:
from sklearn.model_selection import train_test_split

In [137]:
X_train, X_test, y_train, y_test = train_test_split(
    enc_inputs, label_t, test_size=0.2, random_state=42, stratify=label_t
)

In [138]:
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader
# 패딩 토큰 -> 토큰의 길이를 채워주기 위한 특수 토큰 
from torch.nn.utils.rnn import pad_sequence

In [139]:
# 딥러닝에서 사용할 데이터를 파이토치에 맞게 변환 Dataset
class TextDataset(Dataset):
    def __init__(self, xs, ys):
        # 독립 변수
        self.xs = xs
        # 종속 변수
        self.ys = ys
    def __len__(self):
        return len(self.xs)

    def __getitem__(self, idx):
        # 특정 위치에 있는 데이터를 되돌려준다. -> 독립 변수, 종속 변수 
        return self.xs[idx], self.ys[idx]

In [140]:
# DataLoader을 이용하여 배치 사이즈를 생성할때 후 처리 과정 
MAX_K = 5

def collate_fn(batch):
    # batch -> [ (xs[0], ys[0]), (xs[1], ys[1]), .... ]
    # zip(*batch) -> [ (xs[0], xs[1], ...), (ys[0], ys[1], ....)  ]
    # 독립 변수와 종속 변수로 데이터를 나눠서 저장 
    xs, ys = zip(*batch)

    # 텍스트 마다 토큰의 길이가 다르므로, 이를 하나의 직사각형 행렬로 맞추는 패딩 수행 

    # 패딩 토큰의 위치를 변수에 저장 (일반적으로 0)
    pad_id = stoi['<PAD>']

    # 1차 토큰 채우기 
    # TextCNN 모델에서 정해진 크기의 돋보기(3개 단어, 4개 단어, 5개 단어)로 구간을 생성
    # 토큰의 길이가 돋보기의 크기보다 작은 경우에는 에러가 발생 
    # 이를 방지 하기 위해서 모든 문장의 토큰의 길이를 5로 지정(부족한 부분은 PAD 토큰으로 채워준다.)
    fixed = []
    for x in xs:
        # x : 토큰화된 문서 [token1, token2, ...]
        if len(x) < MAX_K:
            # 여기에서 필요한 패딩 토큰의 개수 
            # 돋보기의 최대 크기 - 토큰화된 데이터의 길이 --> 필요한 패딩 토큰의 개수
            need = MAX_K - len(x)

            # 부족한 개수만큼 패딩 토큰을 x에 채워준다. 
            x = torch.cat(
                [
                    x, 
                    torch.full( (need, ), pad_id, dtype=torch.long )
                ]
            )
        fixed.append(x)
    
    # 2차 패딩 토큰 채우기 
    # 전체적인 문서의 길이를 동일하게 맞추기 위한 패딩 토큰 작업 
    # pad_sequence() 함수는 리스트 안에서 가장 긴 문장의 길이를 찾아서 나머지 짧은 문장들의 빈 공간을 패딩 토큰으로 채워주는 기능
    xs_pad = pad_sequence(
        fixed, 
        batch_first= True,      # 첫번째 차원을 배치 크기로 설정 (데이터크기 : [배치크기, 최대문장길이])
        padding_value= pad_id
    )

    # 패딩 처리가 완료 된 후 각 문장들의 실제 길이( 유효 데이터 길이 )를 저장(모델 연산 사용)
    lengths = torch.tensor(
        [ len(x) for x in fixed ], dtype=torch.long
    )

    # 결과 값은 문장 행렬, 라벨 텐서, 문장 길이
    return xs_pad, torch.stack(ys), lengths

In [141]:
train_loader = DataLoader(
    TextDataset(X_train, y_train), 
    batch_size= 8, 
    shuffle = True, 
    collate_fn= collate_fn
)

val_loader = DataLoader(
    TextDataset(X_test, y_test), 
    batch_size = 8, 
    shuffle = True, 
    collate_fn= collate_fn
)

In [142]:
# 학습 딥 러닝 모델을 생성 (kim CNN)
    # nn.Embdding() -> Conv1D(k = 3, 4, 5) -> max-over-time(해당 구간에서 가장 연관이 높은 구간 선택) -> 
    # concat() -> dropout() -> Linear

class TextCNN(nn.Module):
    def __init__(
            self, vocab_size, emb_dim, num_classes, 
            kernel_size = (3, 4, 5), 
            num_channel = 100, 
            pad_idx = 0, 
            dropout = 0.5
    ):
        # vocab_size : 단어 사전의 길이
        # emb_dim : 임베딩 벡터의 차원의 수 
        # num_classes : 분류 class의 개수
        # kernel_size : 묶이는 단어의 개수 목록
        # num_channel : 합성곱 작업 후 출력 차원의 수
        # pad_idx : 패딩토큰의 id 값
        # dropout : 소실되는 데이터 차원의 비율
        super().__init__()
        # 임베딩 가중치 행렬 생성
        self.emb = nn.Embedding(
            vocab_size, emb_dim, padding_idx= pad_idx
        )

        # 합성곱 신경망 모델 전용 리스트를 생성 
        self.convs = nn.ModuleList(
            [
                # 1차 합성곱 신경망 생성(반복문을 이용해서) : 텍스트를 한 방향으로 훑은 방법
                nn.Conv1d(
                    in_channels= emb_dim,   # 입력 채널의 수 : 임베딩 벡터의 차원의 수
                    out_channels= num_channel,      # 출력 채널의 수 : 특징을 몇명에게 물어보고 답을 받을 것인가?(차원의 수)
                    kernel_size= k              # 단어의 구간 설정
                )
                for k in kernel_size
            ]
        )

        # 100차원의 모델이 3개 생성이 되고 출력들을 단순 열 결합 
        # 300차원(과적합의 위험성) -> 일부의 데이터를 소실(0으로 만든다.)
        self.dropout = nn.Dropout(dropout)
        # 선형 모델을 이용하여 분류의 형태로 0, 1의 확률을 출력 
        self.fc = nn.Linear(num_channel * len(kernel_size), num_classes)

        # 기울기 초기화
        self._init_weights()
    def _init_weights(self):
        # 임베딩 벡터 기울기 초기화 : 자비에르
        nn.init.xavier_uniform_(self.emb.weight)
        nn.init.xavier_uniform_(self.fc.weight)
        # 합성곱 모델의 기울기 초기화 
        for conv in self.convs:
            # 비선형 구조에서 사용하는 다중 퍼셉트론 ReLU()를 이용하는 경우 
            # 학습이 안정되도록 사용하는 초기화 
            nn.init.kaiming_uniform_(conv.weight)
    
    # 순전파 함수 생성 
    def forward(self, x):
        # x : DataLoader를 통해서 드러오는 데이터셋 -> 문장 행렬, 라벨 텐서, 문장 길이
        x = self.emb(x)
        # x는 배치크기, 시퀀스의 길이, 임베딩 차원의 수 --> 
        # Conv1d()에 데이터를 입력하기 위해서는 배치 크기, 임베딩 차원의 수, 시퀀스의 길이 
        # 크기의 순서를 변경하는 함수 transpose()
        x = x.transpose(1, 2)

        feat_map = []
        # Conv1d 모델 -> 비선형 함수 -> Max 최대값 생성 -> feat_map 추가 
        for conv in self.convs:
            # conv : Conv1d 모델
            h = torch.relu( conv(x) )  # 배치의 크기, relu 차원의 개수, T` : 시퀀스의 길이 - 커널의 사이즈 + 1
            # h에서 T`의 최대값 
            h = torch.max(h, dim=2).values     # 배치의 크기, relu 차원의 개수
            feat_map.append(h)
        
        # 열을 기준으로 feat_map 단순 결합 
        z = torch.cat( feat_map, dim = 1 )  # 배치의 크기, relu 차원의 수 * len(self.conv)
        # 과적합 방지를 위해서 일부의 데이터를 소실 
        z = self.dropout(z)
        # 선형 모델에 대입하여 예측
        logits = self.fc(z)

        return logits


In [143]:
# 모델 생성 
model = TextCNN(
    vocab_size=len(vocab), 
    emb_dim=128, 
    num_classes=2, 
    kernel_size=(3,4,5), 
    num_channel= 64, 
    pad_idx=stoi['<PAD>'], 
    dropout=0.5
)

In [144]:
# 옵티마이저 설정 
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-03)
# 손실 함수 설정  (예측값과 실젯값 비교 )
criterion = nn.CrossEntropyLoss()

In [145]:
# 학습, 예측을 하는 함수 선언 
def run_epoch(loader, train=True):
    # loader : model에서 사용할 데이터 셋
    # train : 학습모드인가 예측모드인가
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    corret = 0
    total = 0

    for x, y, lengths in loader:
        # 자동 미분을 활성화 할것인가를 train 매개변수로 설정
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            # 학습 모드라면 옵티마이저, 백워드, 스탭
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += float( loss.item() ) * x.size(0)
        # 예측 값 -> [ 확률, 확률 ] -> 높은 확률의 위치
        preds = logits.argmax(dim=1)
        corret += int( (preds == y).sum().item() )
        total += x.size(0)
    mean_loss = total_loss / total
    acc = corret / total
    return mean_loss, acc

In [146]:
for epoch in range(10):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader, False)
    print(f"""
        Epoch : {epoch+1} 
            Train Loss : {round(tr_loss, 4)}, Train Acc : {round(tr_acc, 2)}
            Validation Loss : {round(val_loss, 4)}, Validation Acc : {round(val_acc, 2)}
    """)


        Epoch : 1 
            Train Loss : 0.6532, Train Acc : 0.59
            Validation Loss : 0.4655, Validation Acc : 0.77
    

        Epoch : 2 
            Train Loss : 0.3467, Train Acc : 0.84
            Validation Loss : 0.545, Validation Acc : 0.74
    

        Epoch : 3 
            Train Loss : 0.1944, Train Acc : 0.92
            Validation Loss : 0.673, Validation Acc : 0.79
    

        Epoch : 4 
            Train Loss : 0.126, Train Acc : 0.94
            Validation Loss : 0.9749, Validation Acc : 0.74
    

        Epoch : 5 
            Train Loss : 0.0911, Train Acc : 0.96
            Validation Loss : 1.1983, Validation Acc : 0.74
    

        Epoch : 6 
            Train Loss : 0.1435, Train Acc : 0.95
            Validation Loss : 1.1682, Validation Acc : 0.73
    

        Epoch : 7 
            Train Loss : 0.1347, Train Acc : 0.95
            Validation Loss : 1.2763, Validation Acc : 0.74
    

        Epoch : 8 
            Train Loss : 0.0924, Train

In [147]:
# 예측 함수 생성 

@torch.no_grad()
def predict(text):
    # text : 감성 평가용 문장
    # 문장을 토큰화
    toks = tokenize(text)
    # 인코딩 
    ids = encode(toks)
    pad_id = stoi['<PAD>']

    # 학습된 모델에서 최대 커널의 크기를 확인 
    max_k = max( [   m.kernel_size[0]  for m in model.convs ] )

    if len(ids) < max_k:
        need = max_k - len(ids)
        ids = torch.cat(
            [
                ids, 
                torch.full(
                    (need, ), pad_id, dtype=torch.long
                )
            ]
        )
    # 모델에 입력으로 데이터를 대입하기 위해 차원 구조를 변경 
    x = ids.unsqueeze(0)
    # print(x.shape)
    logits = model(x)
    prob = torch.softmax(logits, dim=1).squeeze(0).tolist()
    pred = int(torch.argmax(logits, dim=1).item())
    return prob, pred

In [148]:
prob, pred = predict("직원의 태도가 별로였고 실망했다")

In [149]:
print("예측 값 : ", pred)
print("예측 확률 : ", prob)

예측 값 :  0
예측 확률 :  [1.0, 1.754569115684997e-19]


In [150]:
# df 하위 100개의 데이터를 이용하여 예측 값을 확인하고 정확도 계산

test_document =  df.tail(100)['document'].values
test_label = df.tail(100)['label'].values

In [151]:
acc = 0

for doc, label in zip(test_document, test_label):
    _, pred = predict(doc)
    if pred == label:
        acc += 1
acc

62

In [152]:
preds = [
    predict(doc)[1] for doc in test_document 
]
preds

[1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0]

In [153]:
(test_label == preds).sum()

np.int64(62)